In [1]:
# Add autoreload at the top of the notebook you're working on 
# in order for it to auto refresh when you change the 'project_package'
%load_ext autoreload
%autoreload 2

# Initialize dummy dataset

In [1]:
import os
import warnings
import threadpoolctl
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from langchain_chroma import Chroma
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker
from rank_bm25 import BM25Plus
from rectools.model_selection import RandomSplitter, cross_validate
from rectools.model_selection.random_split import RandomSplitter
from rectools.models import load_model,ImplicitItemKNNWrapperModel,ImplicitALSWrapperModel,LightFMWrapperModel,ImplicitBPRWrapperModel
from rectools.models.pure_svd import PureSVDModel
import implicit
from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender
from implicit.als import AlternatingLeastSquares as CPU_AlternatingLeastSquares
from implicit.gpu.als import AlternatingLeastSquares as GPU_AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking as CPU_BayesianPersonalizedRanking
from implicit.gpu.bpr import BayesianPersonalizedRanking as GPU_BayesianPersonalizedRanking
from lightfm import LightFM

from project_package.data_preprocessing.utils import create_user_preference,chroma_filter_operator
from project_package.modeling.recommendation_utils import (
    preprocessing_docs,generate_metric_objs,construct_rec_train_dataset,get_embedding_model,
    VectorstoreLoader
    )

load_dotenv()  # load env variables from .evn
root_directory = Path(os.getcwd()).parent.parent  #NOTE: update of notebook location changed


g:\Python\envs\capstone_test3\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


Load item-content metadata

In [2]:
movie_metadata = pd.read_csv(
    root_directory / 'data/external/movielense_25m/movies_metadata.csv',
    usecols = ['id','original_title','overview','genres','original_language',
               'adult','runtime','revenue','vote_average','vote_count']
    )
movie_metadata['genres'] = np.array(movie_metadata['genres'].apply(lambda x:[item['name'] for item in eval(x)]))

movie_metadata['id'] = pd.to_numeric(movie_metadata['id'], errors='coerce')
movie_metadata.dropna(subset=['id'],inplace=True)
movie_metadata['id'] = movie_metadata['id'].astype(int)
movie_metadata.rename(columns={'id':'movieId'},inplace=True)

movie_metadata = movie_metadata.drop_duplicates('movieId').reset_index(drop=True)

movie_metadata['genres'] = movie_metadata['genres'].apply(
    lambda x: ['Unknown'] if isinstance(x, list) and len(x) == 0 else x
)

map_dict = {"False":"People of all age","True":"Adult only"}
movie_metadata['adult'] = movie_metadata['adult'].map(map_dict)
movie_metadata.head(3)

,adult,genres,movieId,original_language,original_title,overview,revenue,runtime,vote_average,vote_count
0,People of all age,"[Animation, Comedy, Family]",862,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",373554033.0,81.0,7.7,5415.0
1,People of all age,"[Adventure, Fantasy, Family]",8844,en,Jumanji,When siblings Judy and Peter discover an encha...,262797249.0,104.0,6.9,2413.0
2,People of all age,"[Romance, Comedy]",15602,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,0.0,101.0,6.5,92.0


Load user rating data

In [3]:
# should sort reviews by user first
user_reviews = pd.read_csv(
    root_directory / 'data/external/movielense_25m/ratings_small.csv',
    usecols = ['userId','movieId','rating'],
    )
user_reviews.head(5)

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


Create user preference data

In [4]:
# Generate user preference data

if not os.path.exists(root_directory / 'data/external/movielense_25m/user_preferences.csv'):
    criteria_dict = dict(
        genres = (0.3,'multiple'),
        original_language = (0.1,'single'),
        adult = (0.4,'single')
    )

    preference_df = create_user_preference(
        movie_metadata,'movieId',
        user_reviews,'userId',
        criteria_dict,
        'rating',
        user_batch=200
    )
    preference_df.to_csv(root_directory / 'data/external/movielense_25m/user_preferences.csv',index =False)
else:
    preference_df = pd.read_csv(root_directory / 'data/external/movielense_25m/user_preferences.csv')

preference_df.head(3)

,userId,genres,original_language,adult
0,1,['Comedy'],['en'],['People of all age']
1,2,['Drama'],['en'],['People of all age']
2,3,['Drama'],"['en', 'fr']",['People of all age']


## 1. Initialize Chroma vectorstore with data

In [5]:
doc_template = """
The movie title:
{}

The movie overview:
{}

The genres:
{}

The movie is for:
{}
"""

user_profile_template = """
Favorite genres:
{}
Favorite languages:
{}
Preferred movie PG type:
{}
"""

embedding_model = get_embedding_model(
    huggingface_model_path="BAAI/bge-small-en-v1.5",  # NOTE: Change this embedding to foodbert later
    local_model_name="bge-small",
    device="cuda"
)
chroma_path = root_directory / "data/external/movielense_25m/chroma_db"  #NOTE: Change the Path later

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Create a collection for item data

In [6]:
#NOTE: Run the cell again when a batch is failed to continue

store_document = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_document:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="movielens_collection",  #NOTE: Change the collection name later,
            embedding = embedding_model,
            doc_template = doc_template,
            input_data = movie_metadata,
            format_cols = ['original_title','overview','genres','adult'],
            meta_cols = ['genres','original_language','adult','runtime','vote_average','vote_count','movieId'],  # include item ID for later filter tasks
            persist_directory = chroma_path,
            docID_col = 'movieId'  #NOTE: Change the ID later
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    vectorstore = Chroma(
        collection_name="movielens_collection",
        embedding_function=embedding_model,
        persist_directory=chroma_path
    )

Create a collection for user preferences

In [7]:
#NOTE: Run the cell again when a batch is failed to continue

store_user_pref = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_user_pref:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="movielens_user_preference",  #NOTE: Change the collection name later,
            embedding = embedding_model,
            doc_template = user_profile_template,
            input_data = preference_df,
            format_cols = ['genres','original_language','adult'],
            meta_cols = None,
            persist_directory = chroma_path,
            docID_col = 'userId'  #NOTE: Change the ID later
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        user_vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    user_vectorstore = Chroma(
        collection_name="movielens_user_preference",
        embedding_function=embedding_model,
        persist_directory=chroma_path
    )

## 2. Train recommendation models 

#### Preparing the training/test datasets and metric format

In [8]:
k = 10  # number of recommendation to create

# load data to Rectools format
dataset = construct_rec_train_dataset(
    user_reviews,
    movie_metadata,
    preference_df,
    use_test_cols=True
)

In [ ]:
# Retrieve item embeddings from Chromastore
# we are using item embedding instead of onehot coded to compare item similarity
ids = vectorstore._collection.get(include=['embeddings'])['ids']  
doc_embeddings = vectorstore._collection.get(include=['embeddings'])['embeddings']
idx = pd.Index(ids,name='item_id',dtype=int)
embedding_docs = pd.DataFrame(doc_embeddings,index=idx)

# create metric objects for k recommendations
metrics = generate_metric_objs(embedding_docs,k=k)

### List of chosen recommendation models 

* **ItemKNN model**

This is a wrapper model for item-item nearest neighbour models. Those models are item-content recommendation models, which based purely on the content and doesn't require user-item interaction or user preferences. A few recommendation model belongs to this type are BM25, TFIDF,etc.

* **SVD model**

This is a basic collaboration model where we only take the user-item score interactions then apply a dimension reduction algorithm to create embedding representation in a latent vector space. This allows the model to generate rating for unseen items and create recommendation to the users.

* **AlternatingLeastSquares**

Goal of the model is to present interactions matrix as a product of user(X) and item(Y) embeddings. Implicit ALS model treats all non-zero entries in the matrix as value. The actual weight of the interactions is treated as confidence in the observation. Zero entries receive low confidence since this they are treated as missing values and might actually hide items highly relevant to users. Non-zero entries with high confidence will have greater impact on the loss when not predicted correctly.

* **BayesianPersonalizedRanking**

Bayesian personalized ranking introduces a pairwise loss instead. For each user model takes a pair of items: one positive and one negative where positive item was present in user interactions and negative item wasn’t. The goal of the algorithm is to rank positive item higher then negative one. It is useful for cases when only positive interactions are present in data and when the goal is to maximize ROC AUC.

* **LightFM**

A hybrid latent representation recommender model.

The model learns embeddings (latent representations in a high-dimensional space) for users and items in a way that encodes user preferences over items. When multiplied together, these representations produce scores for every item for a given user; items scored highly are more likely to be interesting to the user.

The user and item representations are expressed in terms of representations of their features: an embedding is estimated for every feature, and these features are then summed together to arrive at representations for users and items

Cross validation all models for comparison

<ins>skip this part if you already train the models</ins>

In [9]:
# For implicit ALS
os.environ["OPENBLAS_NUM_THREADS"] = "1"
threadpoolctl.threadpool_limits(1, "blas")

tfidf_model = ImplicitItemKNNWrapperModel(TFIDFRecommender())
bm25_model = ImplicitItemKNNWrapperModel(BM25Recommender(K1=1.5))
svd_model = PureSVDModel(factors=100,use_gpu=True)

if implicit.gpu.HAS_CUDA:
    als_model = ImplicitALSWrapperModel(GPU_AlternatingLeastSquares(factors=100,random_state=0))
    bpr_model = ImplicitBPRWrapperModel(GPU_BayesianPersonalizedRanking(factors=100,random_state=0))
else:
    als_model = ImplicitALSWrapperModel(CPU_AlternatingLeastSquares(factors=100,random_state=0))
    bpr_model = ImplicitBPRWrapperModel(CPU_BayesianPersonalizedRanking(factors=100,random_state=0))

#NOTE: Update the loss function when computer support thread
lightfm_model = LightFMWrapperModel(LightFM(no_components=50, loss="logistic",random_state=0))  

models = {
    "TFIDFRecommender":tfidf_model,
    "BM25Recommender":bm25_model,
    "PureSVDModel":svd_model,
    "AlternatingLeastSquares":als_model,
    "BayesianPersonalizedRanking":bpr_model,
    "LightFM":lightfm_model
}

splitter = RandomSplitter(test_fold_frac=0.195, random_state=0,n_splits=5)  # test_fold_frac * n_splits can only be close to 100%, otherwise it's impossible to split

g:\Python\envs\capstone_test3\Lib\site-packages\rectools\models\pure_svd.py:113: UserWarning: Forced to use CPU. CuPy is not available.
  warnings.warn("Forced to use CPU. CuPy is not available.")


In [22]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cv_results = cross_validate(
        dataset=dataset,
        splitter=splitter,
        models=models,
        metrics=metrics,
        k=k,
        filter_viewed=True,
    )

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
cross_validate_df = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
cross_validate_df.columns = cross_validate_df.columns.droplevel(1)
cross_validate_df

,Recall@10,Precision@10,NDCG@10,Novelty@10,AvgRecPopularity@10,Diversity@10,Serendipity@10
model,,,,,,,
TFIDFRecommender,0.145784,0.271993,0.266653,2.766984,0.001597,0.363286,0.001010
BM25Recommender,0.120878,0.223436,0.208393,3.233790,0.001018,0.368346,0.001586
PureSVDModel,0.150102,0.251124,0.245405,2.841890,0.001356,0.363002,0.001401
AlternatingLeastSquares,0.140466,0.241359,0.233323,2.350445,0.001848,0.366417,0.000869
BayesianPersonalizedRanking,0.117044,0.211574,0.205078,3.697660,0.000860,0.363111,0.001664
LightFM,0.000000,0.000000,0.000000,9.390169,0.000000,0.336686,0.000000


Retraining chosen models

In [10]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    svd_model = PureSVDModel(factors=100,use_gpu=True)

    if implicit.gpu.HAS_CUDA:
        als_model = ImplicitALSWrapperModel(GPU_AlternatingLeastSquares(factors=100,random_state=0))
    else:
        als_model = ImplicitALSWrapperModel(CPU_AlternatingLeastSquares(factors=100,random_state=0))

    lightfm_model = LightFMWrapperModel(LightFM(no_components=50, loss="logistic",random_state=0))  

    svd_model.fit(dataset)
    svd_model.save(root_directory / "models/movielens_test/svd_recommendation_model.pkl")  #NOTE: Update Path later
    als_model.fit(dataset)
    als_model.save(root_directory / "models/movielens_test/als_recommendation_model.pkl")  #NOTE: Update Path later
    lightfm_model.fit(dataset)
    lightfm_model.save(root_directory / "models/movielens_test/lightFM_recommendation_model.pkl")  #NOTE: Update Path later

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

## 3. UI implementation

Load the trained recommendation models

In [11]:
#NOTE: Load the models, for real dataset, we might need to load this from S3 or google drive
svd_model = load_model(root_directory / "models/movielens_test/svd_recommendation_model.pkl")
als_model = load_model(root_directory / "models/movielens_test/als_recommendation_model.pkl")
lightfm_model = load_model(root_directory / "models/movielens_test/lightFM_recommendation_model.pkl")

### 3.1. Candidate selection

In [12]:
# create dummy filter component values

test_dict = pd.DataFrame(dict(
    filter_type = ["filter_checklist","filter_dropdown","filter_slider","filter_slider"],
    filter_name = ['genres','original_language_na','vote_average_na','vote_count'],
    filter_value = [['Animation', 'Comedy', 'Family'],['en'],[3,5],[1000,10000]],
    priority_type = ['priority','exact','priority','priority']
))

operator_type_mapping = dict(
    filter_name = ["genres","original_language_na","vote_average_na","vote_count"],
    # record_type = ['list','string','number','number'],
    operator_type = ['$or:$contains','$in','$range','$range']
)

# map UI feature names to database name if they are different
name_mapping = {
    "original_language_na" : "original_language",
    "vote_average_na" : "vote_average"
}


In [13]:
filter_operators = chroma_filter_operator(test_dict,operator_type_mapping,name_mapping)

retriever = vectorstore.as_retriever(
    search_kwargs = dict(
        k=1000,  # We retrieve the best 1000 results
        filter=filter_operators
    )
)

test_query = "story about toy and buzz lightyear"

retrieved_items =  retriever.invoke(test_query)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

candidate_ids = [item.id for item in retrieved_items]

result_docs = [item.page_content for item in retrieved_items]

retrieved_items[:5]

[Document(id='862', metadata={'movieId': 862, 'runtime': 81.0, 'genres': ['Animation', 'Comedy', 'Family'], 'adult': 'People of all age', 'original_language': 'en', 'vote_average': 7.7, 'vote_count': 5415.0}, page_content="\nThe movie title:\nToy Story\n\nThe movie overview:\nLed by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.\n\nThe genres:\n['Animation', 'Comedy', 'Family']\n\nThe movie is for:\nPeople of all age\n"),
 Document(id='863', metadata={'original_language': 'en', 'vote_average': 7.3, 'runtime': 92.0, 'genres': ['Animation', 'Comedy', 'Family'], 'vote_count': 3914.0, 'movieId': 863, 'adult': 'People of all age'}, page_content="\nThe movie title:\nToy Story 2\n\nThe movie overview:\nAndy heads off to Cowboy Camp, leaving his to

### 3.2. Recommendation filtering  & Reranking

This step is used to further choosing the top recommendations that match the user preferences and query content before re-ranking

### 3.2.1. Item-content filtering

We can't use the BM25Recommender model in Rectools for production, because it still need user ratings, but in the UI, we don't actually have user ratings for new users (cold start), so we need to build a pipeline that doesn't utilize user ratings to recommend.

We will be using the BM25Plus model for matching item-contents between the query and the documents

In [14]:
tokenized_corpus = preprocessing_docs(result_docs)
tokenized_query = preprocessing_docs(test_query)

#BM vectorizer model
bm25 = BM25Plus(tokenized_corpus)

doc_scores = bm25.get_scores(tokenized_query)

top_100 = np.array(candidate_ids)[np.argsort(doc_scores)[::-1][:100]]  # sort the similarity score descendingly
top_100_docs = vectorstore.get_by_ids(top_100)
top_100_docs[:5]

[Document(id='862', metadata={'adult': 'People of all age', 'vote_average': 7.7, 'vote_count': 5415.0, 'original_language': 'en', 'runtime': 81.0, 'genres': ['Animation', 'Comedy', 'Family'], 'movieId': 862}, page_content="\nThe movie title:\nToy Story\n\nThe movie overview:\nLed by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.\n\nThe genres:\n['Animation', 'Comedy', 'Family']\n\nThe movie is for:\nPeople of all age\n"),
 Document(id='29911', metadata={'genres': ['Fantasy', 'Drama', 'Mystery', 'Family'], 'original_language': 'en', 'adult': 'People of all age', 'vote_count': 25.0, 'runtime': 99.0, 'vote_average': 6.2, 'movieId': 29911}, page_content="\nThe movie title:\nFairyTale: A True Story\n\nThe movie overview:\nTwo children in 1917 ta

Rerank the documents

In [15]:
# Need to remapping the document ID because using FlashrankRerank the document id in the result will be overwriten with index from 0->n
doc_ids_map = dict(
    zip(range(len(top_100)),top_100.tolist())
)

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)
compressor = FlashrankRerank(client=reranker,top_n=k)  #NOTE: can change to get top 10 rank

In [16]:
rerank_result = compressor.compress_documents(
    top_100_docs,
    query = test_query + ", the movie should be Science Fiction"  #NOTE: can add extra condition here to the query
)

for doc in rerank_result:  # reranking override the doc id so need to change it bank
    doc.metadata['id'] = doc_ids_map[doc.metadata['id']]

data = []

for doc in rerank_result:   # your list of Document objects
    row = doc.metadata.copy()
    row["page_content"] = doc.page_content
    data.append(row)

pd.DataFrame(data)


,id,relevance_score,adult,vote_average,vote_count,original_language,runtime,genres,movieId,page_content
0,862,0.993503,People of all age,7.7,5415.0,en,81.0,"[Animation, Comedy, Family]",862,\nThe movie title:\nToy Story\n\nThe movie ove...
1,84152,0.984221,People of all age,7.3,3914.0,en,92.0,"[Animation, Comedy, Family]",863,\nThe movie title:\nToy Story 2\n\nThe movie o...
2,77887,0.937130,People of all age,7.6,4710.0,en,103.0,"[Animation, Family, Comedy]",10193,\nThe movie title:\nToy Story 3\n\nThe movie o...
3,253150,0.892690,People of all age,5.0,173.0,en,118.0,"[Fantasy, Comedy, Science Fiction]",11597,\nThe movie title:\nToys\n\nThe movie overview...
4,148511,0.628799,People of all age,5.1,11.0,en,90.0,"[Horror, Science Fiction]",70984,"\nThe movie title:\nSilent Night, Deadly Night..."
5,86297,0.452504,People of all age,4.2,12.0,en,64.0,"[Horror, Thriller, Science Fiction, Action, Fa...",84152,\nThe movie title:\nDollman vs. Demonic Toys\n...
6,102431,0.157162,People of all age,7.3,246.0,en,22.0,"[Animation, Comedy, Family]",213121,\nThe movie title:\nToy Story of Terror!\n\nTh...
7,213121,0.099905,People of all age,6.2,522.0,en,110.0,"[Comedy, Adventure, Fantasy, Science Fiction, ...",11551,\nThe movie title:\nSmall Soldiers\n\nThe movi...
8,15302,0.085692,People of all age,5.7,64.0,en,102.0,"[Comedy, Family]",23805,\nThe movie title:\nThe Toy\n\nThe movie overv...
9,279598,0.044800,People of all age,6.8,249.0,en,22.0,"[Animation, Family]",256835,\nThe movie title:\nToy Story That Time Forgot...


### 3.2.2. Collaboration filtering

For this section, we are using SVD model, which only relying on user-item rating interaction to create the recommendation. A drawback of this method is that without any initial ratings for new user, we can't make a recommendation for them. Therefore, to address this problem. We are going to have user input some initial preferences or try to figure out their short-term preference from their filter choices.

In [17]:
test_user_profile = user_profile_template.format(
    ['Drama','Thriller'],
    ['en', 'fr'],
    ['People of all age']
)

print(test_user_profile)


Favorite genres:
['Drama', 'Thriller']
Favorite languages:
['en', 'fr']
Preferred movie PG type:
['People of all age']



We try to retrieve the top user profiles that similar to the user

In [18]:
user_retriever = user_vectorstore.as_retriever(
    search_kwargs = dict(
        k=10,  # We retrieve the 10 best similar users
    )
)

retrieved_users =  user_retriever.invoke(test_user_profile)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

match_user_ids = np.array([item.id for item in retrieved_users],dtype=int)

retrieved_users[:5]

[Document(id='46', metadata={}, page_content="\nFavorite genres:\n['Drama', 'Thriller']\nFavorite languages:\n['en', 'fr']\nPreferred movie PG type:\n['People of all age']\n"),
 Document(id='228', metadata={}, page_content="\nFavorite genres:\n['Drama', 'Thriller']\nFavorite languages:\n['en', 'fr']\nPreferred movie PG type:\n['People of all age']\n"),
 Document(id='329', metadata={}, page_content="\nFavorite genres:\n['Drama', 'Thriller']\nFavorite languages:\n['en', 'fr']\nPreferred movie PG type:\n['People of all age']\n"),
 Document(id='343', metadata={}, page_content="\nFavorite genres:\n['Drama', 'Thriller']\nFavorite languages:\n['en', 'fr']\nPreferred movie PG type:\n['People of all age']\n"),
 Document(id='410', metadata={}, page_content="\nFavorite genres:\n['Drama', 'Thriller']\nFavorite languages:\n['en', 'fr']\nPreferred movie PG type:\n['People of all age']\n")]

We find the top recommendations for the top users that are similar to the test user

In [23]:
model_recommendations = als_model.recommend(
    match_user_ids,
    dataset,
    k=k,
    filter_viewed=False
)
model_recommendations.head(3)

,user_id,item_id,score,rank
0,46,58559,1.886147,1
1,46,4993,1.764741,2
2,46,5952,1.509153,3


Finally we us a weighted random selection of recommendations from the pool of recommendations for similar users

In [21]:
# The strategy is to count how many time a recommendation has been suggested for each user, then we have a weighted shuffle
# to select the k recommendations

counts = model_recommendations.item_id.value_counts()

# Exact IDs and the weights
items = counts.index.values
weights = counts.values

# Normalize the weights
probabilities = weights / weights.sum()

np.random.seed(0)  # can turn seed on/off

recommend_ids = np.random.choice(
    items, 
    size=k, 
    replace=False, 
    p=probabilities
)

recommend_ids

array([  780,  1198,  2571,  8961,  5952,   589, 91529, 58559,   344,
        1073], dtype=int64)

### 3.2.3. Hybrid filtering

With normal collaboration models, many of them won't be able to handle cold-start for items without ratings or user without any reviews. Hybrid models are developed to address this problem, one of them is LightFM.

In [ ]:
# retriever = vectorstore.as_retriever(
#     search_kwargs = dict(
#         k=1000,  # We retrieve the best 1000 results
#         filter={"movieId": {"$in": user_reviews.movieId.unique().tolist()}},  # limit to only items that has been rated.
#     )
# )

# test_query = "story about toy and buzz lightyear"

# retrieved_items =  retriever.invoke(test_query)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

# candidate_ids = [item.id for item in retrieved_items]

In [ ]:
# we are using the same test query
test_query = "story about toy and buzz lightyear"
# and candidate retrieved from vectorstore
candidate_ids[:5]

['862',
 '863',
 '23805',
 '10193',
 '256835',
 '11597',
 '118727',
 '213121',
 '340223',
 '108869']

Using the list of candidate, we input that to the lightFM model to rank the top 100 recommendations for each similar users. Note that because the user preference has been taken into context, the top recommendations in the step might have derived from the text_query

In [61]:
recommendations = lightfm_model.recommend(
    users=match_user_ids,  # we also reuse similar users as we don't have rating for new user yet
    dataset=dataset,
    k=100,
    items_to_recommend=np.array(candidate_ids,dtype=int), # Can contain either hot or warm items
    filter_viewed = False
)

In [89]:
# them we sum the score of each recommended items and sort them
top_100 = recommendations.groupby('item_id')['score'].sum().sort_values(ascending=False).index[:100].to_numpy()
top_100[:5]
top_100_docs = vectorstore.get_by_ids(top_100.astype(str))
top_100_docs[:5]

[Document(id='15789', metadata={'vote_average': 6.7, 'genres': ['Romance', 'Animation', 'Family', 'Comedy', 'Adventure'], 'adult': 'People of all age', 'runtime': 78.0, 'original_language': 'en', 'vote_count': 404.0, 'movieId': 15789}, page_content="\nThe movie title:\nA Goofy Movie\n\nThe movie overview:\nThough Goofy always means well, his amiable cluelessness and klutzy pratfalls regularly embarrass his awkward adolescent son, Max. When Max's lighthearted prank on his high-school principal finally gets his longtime crush, Roxanne, to notice him, he asks her on a date. Max's trouble at school convinces Goofy that he and the boy need to bond over a cross-country fishing trip like the one he took with his dad when he was Max's age, which throws a kink in his son's plans to impress Roxanne.\n\nThe genres:\n['Romance', 'Animation', 'Family', 'Comedy', 'Adventure']\n\nThe movie is for:\nPeople of all age\n"),
 Document(id='11310', metadata={'vote_count': 103.0, 'genres': ['Action', 'Comed

Then the final step would be to rerank the list recommendation from previous step using the text_query context

In [90]:
# Need to remapping the document ID because using FlashrankRerank the document id in the result will be overwriten with index from 0->n
doc_ids_map = dict(
    zip(range(len(top_100)),top_100.tolist())
)

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)
compressor = FlashrankRerank(client=reranker,top_n=k)  #NOTE: can change to get top 10 rank

In [91]:
rerank_result = compressor.compress_documents(
    top_100_docs,
    query = test_query #NOTE: can add extra condition here to the query
)

for doc in rerank_result:  # reranking override the doc id so need to change it bank
    doc.metadata['id'] = doc_ids_map[doc.metadata['id']]

data = []

for doc in rerank_result:   # your list of Document objects
    row = doc.metadata.copy()
    row["page_content"] = doc.page_content
    data.append(row)

pd.DataFrame(data)


,id,relevance_score,adult,original_language,vote_count,vote_average,genres,movieId,runtime,page_content
0,68773,0.000616,People of all age,en,12.0,4.2,"[Horror, Thriller, Science Fiction, Action, Fa...",84152,64.0,\nThe movie title:\nDollman vs. Demonic Toys\n...
1,105945,0.000053,People of all age,en,58.0,6.1,"[Family, Fantasy, Adventure, Science Fiction, ...",13764,108.0,\nThe movie title:\nSanta Claus: The Movie\n\n...
2,84152,0.000043,People of all age,en,525.0,4.7,"[Action, Adventure, Comedy, Family, Science Fi...",12279,84.0,\nThe movie title:\nSpy Kids 3-D: Game Over\n\...
3,332,0.000034,People of all age,en,21.0,5.0,"[Fantasy, Adventure, Comedy, Family]",32643,88.0,\nThe movie title:\nThe Love Bug\n\nThe movie ...
4,11044,0.000033,People of all age,en,16.0,5.5,"[Action, Comedy, Crime, Science Fiction]",68773,82.0,\nThe movie title:\nDollman\n\nThe movie overv...
5,2300,0.000033,People of all age,en,522.0,6.2,"[Comedy, Adventure, Fantasy, Science Fiction, ...",11551,110.0,\nThe movie title:\nSmall Soldiers\n\nThe movi...
6,32834,0.000030,People of all age,en,404.0,6.7,"[Romance, Animation, Family, Comedy, Adventure]",15789,78.0,\nThe movie title:\nA Goofy Movie\n\nThe movie...
7,136793,0.000027,People of all age,en,21.0,5.4,"[Adventure, Drama, Family, Science Fiction]",187462,86.0,\nThe movie title:\nRobosapien: Rebooted\n\nTh...
8,9975,0.000027,People of all age,en,454.0,6.8,"[Science Fiction, Comedy, Drama, Crime]",84329,85.0,\nThe movie title:\nRobot & Frank\n\nThe movie...
9,18974,0.000026,People of all age,en,94.0,5.8,"[Fantasy, Comedy, Science Fiction, Family]",10208,87.0,\nThe movie title:\nMuppets from Space\n\nThe ...
